# Permission-Aware Vector Search: embedding job (Kaggle, T4 GPU)

**What this notebook does**
1. Reads the arXiv metadata dump (Kaggle dataset `Cornell-University/arxiv`, CC0 licence).
2. Draws a fixed random sample: 1,000,000 documents (the search corpus) and 2,000 *different*
   documents whose titles become test queries. The seed is fixed, so the sample is reproducible.
3. Turns each document (title + abstract) into a 384-number embedding with
   `all-MiniLM-L6-v2`, on the GPU in half precision (float16).
4. Saves everything to `/kaggle/working`, plus a 100k slice for the first demo.

**Why on Kaggle:** the same job on the laptop CPU would take hours; a free T4 GPU does it in minutes.
Access labels are NOT assigned here. That happens later in Spark on the laptop, so permissions
live in the lakehouse, not inside this notebook.

In [ ]:
!pip -q install "sentence-transformers==6.2.0"

In [ ]:
import glob, json, os, re, time
import numpy as np
import pandas as pd
import torch
import sentence_transformers
from sentence_transformers import SentenceTransformer

N_CORPUS = int(os.environ.get("N_CORPUS", 1_000_000))
N_QUERY = int(os.environ.get("N_QUERY", 2_000))
N_DEMO = int(os.environ.get("N_DEMO", 100_000))
SEED = 42
MODEL = "sentence-transformers/all-MiniLM-L6-v2"
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

src = os.environ.get("SRC") or (glob.glob("/kaggle/input/**/arxiv-metadata-oai-snapshot.json", recursive=True) or [None])[0]
assert src, "Attach the Kaggle dataset Cornell-University/arxiv to this notebook"
print("source:", src)
print("torch", torch.__version__, "| sentence-transformers", sentence_transformers.__version__, "| device", DEVICE,
      "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

In [ ]:
# Read only the fields we need, one JSON line at a time (the file is ~4-5 GB).
# The dump contains some arXiv ids more than once; we keep the LAST record for each id
# (the most recent metadata) and count how many duplicates were dropped (goes in the data card).
ws = re.compile(r"\s+")
ids, titles, abstracts, cats, dates = [], [], [], [], []
seen = {}
lines = dups = 0
t = time.time()
with open(src, encoding="utf-8") as f:
    for line in f:
        lines += 1
        d = json.loads(line)
        aid = d["id"].strip()
        row = (ws.sub(" ", d.get("title") or "").strip(),
               ws.sub(" ", d.get("abstract") or "").strip(),
               (d.get("categories") or "").strip(),
               d.get("update_date") or "")
        if aid in seen:
            dups += 1
            i = seen[aid]
            titles[i], abstracts[i], cats[i], dates[i] = row
            continue
        seen[aid] = len(ids)
        ids.append(aid)
        titles.append(row[0]); abstracts.append(row[1]); cats.append(row[2]); dates.append(row[3])
del seen
total = len(ids)
read_s = time.time() - t
print(f"read {lines:,} lines in {read_s:.0f}s -> {total:,} unique ids ({dups:,} duplicate records dropped)")

In [ ]:
# Random, reproducible split. Corpus and queries never overlap, so a query can't "find itself".
rng = np.random.default_rng(SEED)
perm = rng.permutation(total)
corpus_rows = perm[:N_CORPUS]                     # random order, so any prefix is also a random sample
query_rows = perm[N_CORPUS:N_CORPUS + N_QUERY]


def frame(rows):
    return pd.DataFrame({
        "arxiv_id": [ids[i] for i in rows],
        "title": [titles[i] for i in rows],
        "abstract": [abstracts[i] for i in rows],
        "categories": [cats[i] for i in rows],
        "update_date": [dates[i] for i in rows],
    })


corpus = frame(corpus_rows)
queries = frame(query_rows)
corpus.insert(0, "doc_id", np.arange(len(corpus), dtype="int64"))
queries.insert(0, "query_id", np.arange(len(queries), dtype="int64"))
del ids, titles, abstracts, cats, dates
assert corpus.arxiv_id.is_unique and not set(corpus.arxiv_id) & set(queries.arxiv_id)
print(len(corpus), "corpus docs |", len(queries), "queries")
print(corpus.categories.str.split().str[0].str.split(".").str[0].value_counts().head(10))

In [ ]:
model = SentenceTransformer(MODEL, device=DEVICE)
if DEVICE == "cuda":
    model.half()
print("max_seq_length:", model.max_seq_length)

texts = (corpus.title + ". " + corpus.abstract).tolist()
t = time.time()
emb = model.encode(texts, batch_size=512, normalize_embeddings=True,
                   convert_to_numpy=True, show_progress_bar=True).astype(np.float16)
corpus_s = time.time() - t
print(f"corpus: {emb.shape} in {corpus_s:.0f}s = {len(texts) / corpus_s:,.0f} docs/s")

t = time.time()
q_emb = model.encode(queries.title.tolist(), batch_size=512, normalize_embeddings=True,
                     convert_to_numpy=True).astype(np.float16)
query_s = time.time() - t
assert emb.shape == (len(corpus), 384) and q_emb.shape == (len(queries), 384)
assert np.isfinite(emb.astype(np.float32)).all()

In [ ]:
os.makedirs(OUT, exist_ok=True)
np.save(f"{OUT}/corpus_emb_f16.npy", emb)
np.save(f"{OUT}/queries_emb_f16.npy", q_emb)
corpus.to_parquet(f"{OUT}/corpus_meta.parquet", compression="zstd", index=False)
queries.to_parquet(f"{OUT}/queries_meta.parquet", compression="zstd", index=False)

# Small slice for the first demo: the first N_DEMO docs are already a random sample.
np.save(f"{OUT}/demo100k_emb_f16.npy", emb[:N_DEMO])
corpus.iloc[:N_DEMO].to_parquet(f"{OUT}/demo100k_meta.parquet", compression="zstd", index=False)

manifest = {
    "source": "Kaggle Cornell-University/arxiv (CC0)", "lines_in_source": lines,
    "unique_ids_in_source": total, "duplicate_records_dropped": dups,
    "n_corpus": len(corpus), "n_queries": len(queries), "n_demo": min(N_DEMO, len(corpus)), "seed": SEED,
    "model": MODEL, "dim": 384, "dtype": "float16", "normalized": True,
    "text_doc": "title + '. ' + abstract", "text_query": "title",
    "device": DEVICE, "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
    "torch": torch.__version__, "sentence_transformers": sentence_transformers.__version__,
    "read_seconds": round(read_s, 1), "corpus_encode_seconds": round(corpus_s, 1),
    "corpus_docs_per_second": round(len(texts) / corpus_s, 1), "query_encode_seconds": round(query_s, 1),
}
json.dump(manifest, open(f"{OUT}/manifest.json", "w"), indent=2)
for f in sorted(os.listdir(OUT)):
    p = os.path.join(OUT, f)
    if os.path.isfile(p):
        print(f"{f:28s} {os.path.getsize(p) / 1e6:8.1f} MB")
print(json.dumps(manifest, indent=2))